In [1]:
import os
import numpy as np
import pandas as pd
import cv2

# import splitfolders
import h5py
from matplotlib import pyplot as plt
%matplotlib inline
from matplotlib import rcParams
import seaborn as sns
from PIL import Image
import imutils 

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.applications import (
    InceptionResNetV2,
    ResNet50,
    InceptionV3,
    DenseNet121,
)
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.imagenet_utils import preprocess_input
from tensorflow.keras.callbacks import TensorBoard
from sklearn.metrics import classification_report, confusion_matrix



In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive/", force_remount=True)
    google_drive_prefix = "/content/drive/My Drive"
    data_prefix = "{}/mnist/".format(google_drive_prefix)
except ModuleNotFoundError: 
    data_prefix = "data/"

Mounted at /content/drive/


In [3]:
!pip install wandb

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.3/184.3 KB 17.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 KB 17.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 KB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 KB 11.4 MB/s eta 0:00:00
  Created wheel for pathtools: filename=pathtools-0.1.2-py3-none-any.whl size=8806 sha256=66b313f1cab77aa437967e8a950ec66fddeff1c316f89686aec843312c78007a
  Stored in directory: /root/.cache/pip/wheels/4c/8e/7e/72fbc243e1aeecae64a96875432e70d4e92f3d2d18123be004
Successfully built pathtools
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.24.3
    Uninstalling urllib3-1.24.3:
      Successfully uninstalled urllib3-1.24.3


In [4]:
import wandb
wandb.login()

ERROR:wandb.jupyter:Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


<IPython.core.display.Javascript object>

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [5]:
from wandb.keras import WandbMetricsLogger, WandbModelCheckpoint

In [19]:
# Start a run, tracking hyperparameters
wandb.init(
    # set the wandb project where this run will be logged
    project="Mermoire_2023_Version_01",

    # track hyperparameters and run metadata with wandb.config
    config={
        "dropout": 0.25,
        "dropout_2": 0.2,
        "activation_1": "relu",
        "activation_2": "softmax",
        "optimizer": "Adam",
        "loss": "categorical_crossentropy",
        "metric": "accuracy",
        "epoch": 20,
        "batch_size": 32,
        "units_1": 128,
        "learning_rate": 0.001
    }
)

batch/accuracy,█▅▄▅▄▄▄▁▅▅▅▅▅▅▄▅▅▄▅▅▁▅▅▄▅▅▅▄▅▅▅▅▅▁▃▄▅▅▅▅
batch/batch_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,▁█▅▄▃▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁
epoch/accuracy,▁████
epoch/epoch,▁▃▅▆█
epoch/learning_rate,▁▁▁▁▁
epoch/loss,█▂▂▁▁
epoch/val_accuracy,▁▁▁▁▁
epoch/val_loss,██▇▂▁
batch/accuracy,0.45739


In [20]:
config = wandb.config

In [21]:
train_set = '/content/drive/My Drive/Datasets/Cropped_Image_Sets/train/'
val_set = '/content/drive/My Drive/Datasets/Cropped_Image_Sets/val/'
test_set = '/content/drive/My Drive/Datasets/Cropped_Image_Sets/test/'
augmented_set = '/content/drive/My Drive/Datasets/Augmented/'
# model_dir ="/content/drive/My Drive/Models/RadImageNet-DenseNet121_notop.h5"
model_dir ="/content/drive/My Drive/Models/RadImageNet-ResNet50_notop.h5"
IMAGE_SIZE = 224

In [22]:
def init_data(train_dir: str, valid_dir: str, test_dir: str) -> list:
    train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale = 1/255,
        # samplewise_center=True,
        # samplewise_std_normalization= True,
        horizontal_flip = True,
        vertical_flip = True,
        width_shift_range = 0.1,
        height_shift_range = 0.1,
        # shear_range = 0.2,
        rotation_range = 5,
        zoom_range = [0.8, 1]
    )
    valid_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255,
        # featurewise_center=True,
        # featurewise_std_normalization= True
    )
    test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255,
        # featurewise_center=True,
        # featurewise_std_normalization= True
    )
    
    train_data = train_datagen.flow_from_directory(
        directory=train_dir,
        save_to_dir=augmented_set,
        classes=['glioma_cropped', 'meningioma_cropped', 'pituitary_tumor_cropped'],
        class_mode='categorical',
        target_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=32,
        seed=22,
        shuffle=True,
    )
    valid_data = valid_datagen.flow_from_directory(
        directory=valid_dir,
        save_to_dir=augmented_set,
        classes=['glioma_cropped', 'meningioma_cropped', 'pituitary_tumor_cropped'],
        class_mode='categorical',
        target_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=32,
        seed=22,
        shuffle=True,
    )
    
    test_data = test_datagen.flow_from_directory(
        directory=test_dir,
        save_to_dir=augmented_set,
        classes=['glioma_cropped', 'meningioma_cropped', 'pituitary_tumor_cropped'],
        class_mode='categorical',
        target_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=32,
        seed=22,
        shuffle=True,
    )
    
    return train_data, valid_data, test_data

In [23]:
train_data, valid_data, test_data = init_data(train_dir=train_set, valid_dir=val_set, test_dir=test_set)

Found 2144 images belonging to 3 classes.
Found 458 images belonging to 3 classes.
Found 472 images belonging to 3 classes.


In [24]:
n_samples_train = len(train_data)*(config.batch_size)
print("Number of samples in the training set: ",n_samples_train)

n_samples_val = len(valid_data)*(config.batch_size)
print("Number of samples in the training set: ",n_samples_val)

n_samples_test = len(test_data)*(config.batch_size)
print("Number of samples in the training set: ",n_samples_test)

Number of samples in the training set:  2144
Number of samples in the training set:  480
Number of samples in the training set:  480


In [26]:
# model_name = "My_model"

# TensorBoard = TensorBoard(log_dir="logs\\{}".format(model_name))

TypeError: ignored

In [28]:
def build_transfer_learning_model(base_model):
    # `base_model` stands for the pretrained model
    # We want to use the learned weights, and to do so we must freeze them
    for layer in base_model.layers:
        layer.trainable = False
        
    # Declare a sequential model that combines the base model with custom layers
    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(rate=config.dropout_2),
        tf.keras.layers.Dense(units=128, activation=config.activation_1),
        tf.keras.layers.Dropout(rate=config.dropout_2),
        tf.keras.layers.Dense(units=3, activation=config.activation_2)
    ])

    # Compile the model
    model.compile(
        loss=config.loss,
        optimizer=Adam(learning_rate=config.learning_rate),
        metrics=[config.metric]
    )
    
    return model

In [ ]:
# rad_model = build_transfer_learning_model(
#     base_model = ResNet50(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),  include_top=False, pooling="avg")
# )
rad_model = build_transfer_learning_model(
    base_model = ResNet50(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),  include_top=False)
)

In [ ]:
rad_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50 (Functional)       (None, 7, 7, 2048)        23587712  
                                                                 
 global_average_pooling2d (G  (None, 2048)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dropout_2 (Dropout)         (None, 2048)              0         
                                                                 
 dense_2 (Dense)             (None, 128)               262272    
                                                                 
 dropout_3 (Dropout)         (None, 128)               0         
                                                                 
 dense_3 (Dense)             (None, 3)                 387       
                                                      

In [18]:
# Train the model for 10 epochs
rad_hist = rad_model.fit(
    train_data,
    validation_data=valid_data,
    steps_per_epoch= n_samples_train/config.batch_size,
    epochs=config.epoch,
    callbacks= [WandbMetricsImage()
                WandbMetricsLogger(log_freq=5),
                WandbModelCheckpoint("models"),
                ]
)
wandb.finish()

Epoch 1/20
67/67 [==============================] - ETA: 0s - loss: 1.7822 - accuracy: 0.3643 

wandb: Adding directory to artifact (./models)... Done. 0.7s


67/67 [==============================] - 956s 14s/step - loss: 1.7822 - accuracy: 0.3643 - val_loss: 1.0613 - val_accuracy: 0.4651
Epoch 2/20
67/67 [==============================] - ETA: 0s - loss: 1.0915 - accuracy: 0.4711 

wandb: Adding directory to artifact (./models)... Done. 0.4s


67/67 [==============================] - 867s 13s/step - loss: 1.0915 - accuracy: 0.4711 - val_loss: 1.0654 - val_accuracy: 0.4651
Epoch 3/20
67/67 [==============================] - ETA: 0s - loss: 1.0509 - accuracy: 0.4655

...

wandb: Adding directory to artifact (./models)... Done. 0.5s


67/67 [==============================] - 851s 13s/step - loss: 0.9774 - accuracy: 0.4655 - val_loss: 0.9001 - val_accuracy: 0.4651
Epoch 5/20
67/67 [==============================] - ETA: 0s - loss: 0.9447 - accuracy: 0.4655 

wandb: Adding directory to artifact (./models)... Done. 0.5s


67/67 [==============================] - 843s 13s/step - loss: 0.9447 - accuracy: 0.4655 - val_loss: 0.8696 - val_accuracy: 0.4651
Epoch 6/20
67/67 [==============================] - ETA: 0s - loss: 0.9478 - accuracy: 0.4655 

KeyboardInterrupt: ignored

In [ ]:
path = "/content/drive/My Drive/Models/"
saved_model = path + "model_03" + "_config.dropout_2" + "_config.learning_rate" + "_config.batch_size" + "_" + config.optimizer + ".h5"
print("Saving: ", saved_model)
rad_model.save(path)

Saving:  /content/drive/My Drive/Models/model_03_config.dropout_2_config.learning_rate_config.batch_size_Adam.h5


In [ ]:
print("Evaluate on test data")
results = rad_model.evaluate(test_data, batch_size=32)
print("test loss, test acc:", results)

Evaluate on test data
15/15 [==============================] - 39s 2s/step - loss: 0.7195 - accuracy: 0.7055
test loss, test acc: [0.7194754481315613, 0.7055084705352783]


In [ ]:
Y_pred = rad_model.predict(test_data, n_samples_test // config.batch_size+1)
y_pred = np.argmax(Y_pred, axis=1)
print('Confusion Matrix')
print(confusion_matrix(test_data.classes, y_pred))
print('Classification Report')
target_names = ['glioma', 'meningioma', 'pituitary']
print(classification_report(test_data.classes, y_pred, target_names=target_names))

15/15 [==============================] - 51s 3s/step
Confusion Matrix
[[196  29   0]
 [ 31  71   5]
 [ 42  32  66]]
Classification Report
              precision    recall  f1-score   support

      glioma       0.73      0.87      0.79       225
  meningioma       0.54      0.66      0.59       107
   pituitary       0.93      0.47      0.63       140

    accuracy                           0.71       472
   macro avg       0.73      0.67      0.67       472
weighted avg       0.74      0.71      0.70       472

